# YOLO Foundations Part 1: Tasks & Inference

Welcome to the first lab for the Ultralytics YOLO Foundations course! In this lab, we will focus exclusively on running **inference** (`predict` mode) across the various tasks YOLO supports.

You will not need to train any models today; instead, you will use pre-trained `.pt` weights that come directly from Ultralytics to see what YOLO can do out of the box.

## 1. Environment Setup
First, let's install the `ultralytics` package and its dependencies.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    %pip install -q ultralytics
else:
    !pip install -q ultralytics

import ultralytics
ultralytics.checks()

## 2. Command Line Interface (CLI) Basics

The YOLO CLI is the easiest way to get started. The basic syntax is `yolo TASK MODE ARGS`.

Let's run a simple **Object Detection** task using the `predict` mode.

In [ ]:
!yolo task=detect mode=predict model=yolo26n.pt source="https://ultralytics.com/images/bus.jpg"

## 3. Python API & Exploring Tasks

While the CLI is great for quick tests, the Python API is what you'll use to build applications.

Below, we will explore the 5 core tasks: **Detection, Classification, Segmentation, Pose, and OBB** using the Python API.

### 3.1 Object Detection
Finds and localizes objects using bounding boxes.

In [ ]:
from ultralytics import YOLO
from PIL import Image

# Load a pre-trained detection model
model = YOLO("yolo26n.pt")

# Run prediction
results = model.predict(source="https://ultralytics.com/images/zidane.jpg")

# Explore the results
result = results[0]
print(f"Detected {len(result.boxes)} objects.")
print(f"Boxes Tensor Shape (N, 6): {result.boxes.data.shape}")

# Show the annotated image inline
Image.fromarray(result.plot()[:,:,::-1])  # Convert BGR to RGB for correct display

### 💡 Exercise: Manual Bounding Box Extraction

Instead of relying on `.plot()`, extract the raw coordinates for the **first detected object** and print its $(x1, y1, x2, y2)$ values.

*Tip: Check result.boxes[0].xyxy*

In [ ]:
# TODO: Extract and print (x1, y1, x2, y2) for result.boxes[0]


<details>
<summary>Solution</summary>

```python
first_box = result.boxes[0]
coords = first_box.xyxy[0].tolist()
print(f"Coordinates: {coords}")
```

</details>

### 3.2 Image Classification
Assigns a single label to the entire image.

In [ ]:
model_cls = YOLO("yolo26n-cls.pt")
results_cls = model_cls.predict(source="https://ultralytics.com/images/bus.jpg")

result = results_cls[0]
print(f"Top-1 Class index: {result.probs.top1}")
print(f"Top-1 Class label: {result.names[result.probs.top1]}")
print(f"Top-1 Confidence: {result.probs.top1conf:.2f}")

Image.fromarray(result.plot()[:,:,::-1])

### 💡 Exercise: Confidence Thresholding

Filter the `probs.data` tensor manually. Print the indices of all classes with a confidence score $> 0.05$.

In [ ]:
# TODO: Find and print indices where prob > 0.05


<details>
<summary>Solution</summary>

```python
import torch
indices = torch.where(result.probs.data > 0.05)[0]
print(f"High confidence indices: {indices.tolist()}")
```

</details>

### 3.3 Instance Segmentation
Outlines the exact shape/pixel mask of each object.

In [ ]:
model_seg = YOLO("yolo26n-seg.pt")
results_seg = model_seg.predict(source="https://ultralytics.com/images/zidane.jpg")

result = results_seg[0]
if result.masks:
    print(f"Masks tensor shape (N, H, W): {result.masks.data.shape}")

Image.fromarray(result.plot()[:,:,::-1])

### 3.4 Pose Estimation
Detects people and highlights key anatomical points (joints/keypoints).

In [ ]:
model_pose = YOLO("yolo26n-pose.pt")
results_pose = model_pose.predict(source="https://ultralytics.com/images/bus.jpg")

result = results_pose[0]
if result.keypoints:
    print(f"Keypoints tensor shape (Persons, Keypoints, Coordinates): {result.keypoints.data.shape}")

Image.fromarray(result.plot()[:,:,::-1])

### 3.5 Oriented Bounding Boxes (OBB)
Detects objects using rotated boxes, ideal for top-down or aerial imagery.

In [ ]:
# Due to yolo26 mostly defaulting to standard tasks, obb might fall back to yolo8/11. 
# Ultralytics auto-downloads the best fit if model string is generic.
model_obb = YOLO("yolo11n-obb.pt") 

# Using a sample aerial image for OBB
results_obb = model_obb.predict(source="https://ultralytics.com/images/boats.jpg")

result = results_obb[0]
if getattr(result, "obb", None) is not None:
    print(f"OBB tensor shape (Boxes, 7): {result.obb.data.shape}")
    print("OBB attributes: cx, cy, w, h, angle, conf, cls")

Image.fromarray(result.plot()[:,:,::-1])

---
## End of Lab
You have successfully completed inference (`predict` mode) across all 5 major YOLO tasks using both the CLI and Python API! 

In Part 2 of this course, we will dive into Custom Data and Training (`train` mode).